## Dataset Loading and preparation

In [ ]:
# Section 1: Imports & Paths
import pandas as pd
import numpy as np
import json, os
import os
import pickle
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold, train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, roc_curve
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
import shap
import optuna
import warnings
from dotenv import load_dotenv
warnings.filterwarnings("ignore")
RESULTS_DIR = "results/"
load_dotenv()



In [41]:
from datasets import load_dataset

DATA_PATH = load_dataset("KathiS/Final_Preprocessed_IoTID20", token=hf_token)

df = DATA_PATH["train"].to_pandas()

print("Dataset Shape:", df.shape)

Using the latest cached version of the dataset since KathiS/Final_Preprocessed_IoTID20 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\Maruf\.cache\huggingface\datasets\KathiS___final_preprocessed_io_tid20\default\0.0.0\48c96054df34f3d762f533743ddfbccf1c314115 (last modified on Fri Nov 21 00:49:38 2025).


Dataset Shape: (625783, 26)


In [42]:
# Section 2: Data Preparation
from sklearn.preprocessing import LabelEncoder

if df['Label'].dtype == 'object':
    le = LabelEncoder()
    df['Label'] = le.fit_transform(df['Label'])

X = df.drop("Label", axis=1)
y = df["Label"]

print("Dataset Shape:", df.shape)
print("Feature Columns:", X.columns.tolist())

Dataset Shape: (625783, 26)
Feature Columns: ['Flow_Duration', 'Dst_Port', 'Flow_IAT_Min', 'Flow_IAT_Mean', 'Flow_IAT_Max', 'Flow_Pkts/s', 'Flow_Byts/s', 'Pkt_Len_Max', 'Pkt_Len_Mean', 'Pkt_Size_Avg', 'Pkt_Len_Min', 'Pkt_Len_Std', 'Idle_Min', 'Idle_Mean', 'Idle_Max', 'Fwd_Pkts/s', 'Bwd_Pkts/s', 'Init_Bwd_Win_Byts', 'ACK_Flag_Cnt', 'SYN_Flag_Cnt', 'Bwd_Header_Len', 'Fwd_Pkt_Len_Max', 'Fwd_Pkt_Len_Min', 'Bwd_Pkt_Len_Max', 'Bwd_Pkt_Len_Mean']


In [43]:
os.makedirs("results", exist_ok=True)

with open("results/label_mapping.json","w") as f:
    json.dump(dict(zip(le.classes_, range(len(le.classes_)))), f, indent=4)



In [44]:
# Section 3: Utility Functions

# 1. Metrics calculation
def compute_metrics(y_true, y_pred):

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    cm = confusion_matrix(y_true, y_pred).tolist()

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "confusion_matrix": cm
    }

# 2. Plot Confusion Matrix
def plot_confusion_matrix(cm, labels, title="Confusion Matrix"):
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=labels, yticklabels=labels, cmap="Blues")
    plt.title(title)
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.show()

# 3. Save model
def save_model(model, filename):
    with open(filename, "wb") as f:
        pickle.dump(model, f)



In [45]:
# Section 4: Optuna Tuning
def optuna_tuning_xgb(X, y, n_trials=30):
    def objective(trial):
        param = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "max_depth": trial.suggest_int("max_depth", 3, 15),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "gamma": trial.suggest_float("gamma", 0, 5)
        }
        kf = KFold(n_splits=3, shuffle=True, random_state=42)
        f1_scores = []
        for train_idx, val_idx in kf.split(X):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
            model = XGBClassifier(**param, use_label_encoder=False, eval_metric='mlogloss')
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            f1_scores.append(f1_score(y_val, y_pred, average='weighted'))
        return np.mean(f1_scores)
    
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)
    return study.best_params

def optuna_tuning_lgb(X, y, n_trials=30):
    def objective(trial):
        param = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "max_depth": trial.suggest_int("max_depth", 3, 15),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
            "num_leaves": trial.suggest_int("num_leaves", 20, 150),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0)
        }
        kf = KFold(n_splits=3, shuffle=True, random_state=42)
        f1_scores = []
        for train_idx, val_idx in kf.split(X):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
            model = LGBMClassifier(**param)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            f1_scores.append(f1_score(y_val, y_pred, average='weighted'))
        return np.mean(f1_scores)
    
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)
    return study.best_params


def optuna_tuning_rf(X, y, n_trials=30):
    def objective(trial):
        param = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "max_depth": trial.suggest_int("max_depth", 3, 30),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5)
        }
        kf = KFold(n_splits=3, shuffle=True, random_state=42)
        f1_scores = []
        for train_idx, val_idx in kf.split(X):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
            model = RandomForestClassifier(**param,n_jobs=-1,)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            f1_scores.append(f1_score(y_val, y_pred, average='weighted'))
        return np.mean(f1_scores)
    
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)
    return study.best_params



In [46]:
def train_models_kfold_collect(X, y, model, model_name, k=5):

    kf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

    predictions = []
    metrics_list = []

    fold = 1

    for train_idx, val_idx in kf.split(X, y):

        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model.fit(X_train, y_train)

        y_pred = model.predict(X_val)

        if hasattr(model, "predict_proba"):
            y_prob = model.predict_proba(X_val)
            confidence = y_prob.max(axis=1)
        else:
            confidence = np.ones(len(y_pred))

        # store per-sample results
        fold_df = X_val.copy()
        fold_df["y_true"] = y_val.values
        fold_df["y_pred"] = y_pred
        fold_df["confidence"] = confidence
        fold_df["fold"] = fold
        fold_df["model"] = model_name

        predictions.append(fold_df)

        metrics = compute_metrics(y_val, y_pred)
        metrics["fold"] = fold
        metrics_list.append(metrics)

        fold += 1

    predictions_df = pd.concat(predictions).reset_index(drop=True)

    return metrics_list, predictions_df

In [47]:
# json serializable conversion
import numpy as np
import json

def make_json_serializable(metrics_all):
    serializable = {}
    for model, folds in metrics_all.items():
        serializable[model] = []
        for fold in folds:
            fold_copy = {}
            for k, v in fold.items():
                if isinstance(v, np.ndarray):
                    fold_copy[k] = v.tolist()  # convert array to list
                elif isinstance(v, (np.float64, np.float32)):
                    fold_copy[k] = float(v)    # convert to float
                elif isinstance(v, (np.int64, np.int32)):
                    fold_copy[k] = int(v)      # convert to int
                else:
                    fold_copy[k] = v
            serializable[model].append(fold_copy)
    return serializable



### Model Training with OPTUNA

In [48]:
# ------------------------------
# RandomForest Hyperparameter Tuning
# ------------------------------

best_rf_params = optuna_tuning_rf(X, y, n_trials=10)

with open("results/rf_best_params.json","w") as f:
    json.dump(best_rf_params,f,indent=4)

# ------------------------------
# K-Fold Evaluation
# ------------------------------

rf_model = RandomForestClassifier(**best_rf_params, n_jobs=-1, random_state=42)

rf_metrics, rf_predictions = train_models_kfold_collect(
    X, y, rf_model, "RandomForest", k=3
)

rf_predictions.to_csv("results/rf_predictions.csv", index=False)

with open("results/rf_metrics.json","w") as f:
    json.dump(rf_metrics,f,indent=4)

# ------------------------------
# Train Final Model on FULL DATA
# ------------------------------

rf_final = RandomForestClassifier(**best_rf_params, n_jobs=-1, random_state=42)

rf_final.fit(X, y)

# Save final model
save_model(rf_final, "results/rf_model.pkl")

[I 2026-03-07 22:18:51,412] A new study created in memory with name: no-name-e9f0f69c-7688-4a2f-aeab-1ffdfc2b9144
[I 2026-03-07 22:20:31,203] Trial 0 finished with value: 0.5291658337594335 and parameters: {'n_estimators': 315, 'max_depth': 7, 'min_samples_split': 5, 'min_samples_leaf': 3}. Best is trial 0 with value: 0.5291658337594335.
[I 2026-03-07 22:23:21,073] Trial 1 finished with value: 0.6425298156947723 and parameters: {'n_estimators': 321, 'max_depth': 24, 'min_samples_split': 6, 'min_samples_leaf': 3}. Best is trial 1 with value: 0.6425298156947723.
[I 2026-03-07 22:25:02,797] Trial 2 finished with value: 0.6384285106536173 and parameters: {'n_estimators': 229, 'max_depth': 17, 'min_samples_split': 3, 'min_samples_leaf': 3}. Best is trial 1 with value: 0.6425298156947723.
[I 2026-03-07 22:26:12,864] Trial 3 finished with value: 0.49983658379116297 and parameters: {'n_estimators': 332, 'max_depth': 4, 'min_samples_split': 7, 'min_samples_leaf': 1}. Best is trial 1 with value:

In [49]:
# ------------------------------
# LightGBM Hyperparameter Tuning
# ------------------------------

best_lgb_params = optuna_tuning_lgb(X, y, n_trials=10)

with open("results/lgb_best_params.json","w") as f:
    json.dump(best_lgb_params,f,indent=4)

# ------------------------------
# K-Fold Evaluation
# ------------------------------

lgb_model = LGBMClassifier(**best_lgb_params, n_jobs=-1, random_state=42)

lgb_metrics, lgb_predictions = train_models_kfold_collect(
    X, y, lgb_model, "LightGBM", k=3
)

lgb_predictions.to_csv("results/lgb_predictions.csv", index=False)

with open("results/lgb_metrics.json","w") as f:
    json.dump(lgb_metrics,f,indent=4)

# ------------------------------
# Train Final Model on FULL DATA
# ------------------------------

lgb_final = LGBMClassifier(**best_lgb_params, n_jobs=-1, random_state=42)

lgb_final.fit(X, y)

# Save final model
save_model(lgb_final, "results/lgb_model.pkl")

[I 2026-03-07 22:39:42,895] A new study created in memory with name: no-name-4279d7f9-e4b8-4dca-9af3-ebbf7d0576f3


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.015829 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5474
[LightGBM] [Info] Number of data points in the train set: 417188, number of used features: 25
[LightGBM] [Info] Start training from score -2.360203
[LightGBM] [Info] Start training from score -2.864230
[LightGBM] [Info] Start training from score -2.428991
[LightGBM] [Info] Start training from score -2.417578
[LightGBM] [Info] Start training from score -1.641350
[LightGBM] [Info] Start training from score -1.225377
[LightGBM] [Info] Start training from score -2.747601
[LightGBM] [Info] Start training from score -3.340939
[LightGBM] [Info] Start training from score -2.471497
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

[I 2026-03-07 22:41:58,508] Trial 0 finished with value: 0.6049455807975431 and parameters: {'n_estimators': 479, 'max_depth': 5, 'learning_rate': 0.18873228536334097, 'num_leaves': 35, 'subsample': 0.9205480084927907, 'colsample_bytree': 0.8727957093754228}. Best is trial 0 with value: 0.6049455807975431.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.013905 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5474
[LightGBM] [Info] Number of data points in the train set: 417188, number of used features: 25
[LightGBM] [Info] Start training from score -2.360203
[LightGBM] [Info] Start training from score -2.864230
[LightGBM] [Info] Start training from score -2.428991
[LightGBM] [Info] Start training from score -2.417578
[LightGBM] [Info] Start training from score -1.641350
[LightGBM] [Info] Start training from score -1.225377
[LightGBM] [Info] Start training from score -2.747601
[LightGBM] [Info] Start training from score -3.340939
[LightGBM] [Info] Start training from score -2.471497
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

[I 2026-03-07 22:42:28,986] Trial 1 finished with value: 0.660114206250369 and parameters: {'n_estimators': 100, 'max_depth': 15, 'learning_rate': 0.14625536365601627, 'num_leaves': 70, 'subsample': 0.780172504196788, 'colsample_bytree': 0.6113407258324806}. Best is trial 1 with value: 0.660114206250369.


...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.026435 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5474
[LightGBM] [Info] Number of data points in the train set: 417188, number of used features: 25
[LightGBM] [Info] Start training from score -2.360203
[LightGBM] [Info] Start training from score -2.864230
[LightGBM] [Info] Start training from score -2.428991
[LightGBM] [Info] Start training from score -2.417578
[LightGBM] [Info] Start training from score -1.641350
[LightGBM] [Info] Start training from score -1.225377
[LightGBM] [Info] Start training from score -2.747601
[LightGBM] [Info] Start training from score -3.340939
[LightGBM] [Info] Start training from score -2.471497
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

[I 2026-03-07 22:55:57,897] Trial 8 finished with value: 0.64803844590983 and parameters: {'n_estimators': 227, 'max_depth': 15, 'learning_rate': 0.1352648776741327, 'num_leaves': 125, 'subsample': 0.6243685955305419, 'colsample_bytree': 0.8841313481018347}. Best is trial 2 with value: 0.6652213351328965.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.024266 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5474
[LightGBM] [Info] Number of data points in the train set: 417188, number of used features: 25
[LightGBM] [Info] Start training from score -2.360203
[LightGBM] [Info] Start training from score -2.864230
[LightGBM] [Info] Start training from score -2.428991
[LightGBM] [Info] Start training from score -2.417578
[LightGBM] [Info] Start training from score -1.641350
[LightGBM] [Info] Start training from score -1.225377
[LightGBM] [Info] Start training from score -2.747601
[LightGBM] [Info] Start training from score -3.340939
[LightGBM] [Info] Start training from score -2.471497
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

[I 2026-03-07 22:57:21,208] Trial 9 finished with value: 0.6177370452306529 and parameters: {'n_estimators': 226, 'max_depth': 13, 'learning_rate': 0.15195154671169867, 'num_leaves': 80, 'subsample': 0.8063185054801071, 'colsample_bytree': 0.7696251653184958}. Best is trial 2 with value: 0.6652213351328965.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.018162 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5484
[LightGBM] [Info] Number of data points in the train set: 417188, number of used features: 25
[LightGBM] [Info] Start training from score -2.354859
[LightGBM] [Info] Start training from score -2.872968
[LightGBM] [Info] Start training from score -2.429426
[LightGBM] [Info] Start training from score -2.416906
[LightGBM] [Info] Start training from score -1.641721
[LightGBM] [Info] Start training from score -1.226488
[LightGBM] [Info] Start training from score -2.748312
[LightGBM] [Info] Start training from score -3.339248
[LightGBM] [Info] Start training from score -2.467334
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.020772 seconds.
You can set `force_col_wise=true` to remove t

In [50]:
# ------------------------------
# XGBoost Hyperparameter Tuning
# ------------------------------

best_xgb_params = optuna_tuning_xgb(X, y, n_trials=10)

with open("results/xgb_best_params.json","w") as f:
    json.dump(best_xgb_params,f,indent=4)

# ------------------------------
# K-Fold Evaluation
# ------------------------------

xgb_model = XGBClassifier(
    **best_xgb_params,
    use_label_encoder=False,
    eval_metric='mlogloss',
    n_jobs=-1,
    random_state=42
)

xgb_metrics, xgb_predictions = train_models_kfold_collect(
    X, y, xgb_model, "XGBoost", k=3
)

xgb_predictions.to_csv("results/xgb_predictions.csv", index=False)

with open("results/xgb_metrics.json","w") as f:
    json.dump(xgb_metrics,f,indent=4)

# ------------------------------
# Train Final Model on FULL DATA
# ------------------------------

xgb_final = XGBClassifier(
    **best_xgb_params,
    use_label_encoder=False,
    eval_metric='mlogloss',
    n_jobs=-1,
    random_state=42
)

xgb_final.fit(X, y)

# Save final model
save_model(xgb_final, "results/xgb_model.pkl")

[I 2026-03-07 22:58:19,919] A new study created in memory with name: no-name-cac4be6c-301d-40cc-b30d-aa1ada47f853
[I 2026-03-07 23:00:47,958] Trial 0 finished with value: 0.6484712587620413 and parameters: {'n_estimators': 358, 'max_depth': 10, 'learning_rate': 0.08890302190019574, 'subsample': 0.7630087632475145, 'colsample_bytree': 0.6923580905715898, 'gamma': 0.42480127382982624}. Best is trial 0 with value: 0.6484712587620413.
[I 2026-03-07 23:02:07,648] Trial 1 finished with value: 0.6623191136001553 and parameters: {'n_estimators': 280, 'max_depth': 6, 'learning_rate': 0.07698089403888024, 'subsample': 0.8720580748725473, 'colsample_bytree': 0.7932389311938219, 'gamma': 2.5771578462220166}. Best is trial 1 with value: 0.6623191136001553.
[I 2026-03-07 23:04:12,264] Trial 2 finished with value: 0.6591852510733597 and parameters: {'n_estimators': 425, 'max_depth': 4, 'learning_rate': 0.028718887135262515, 'subsample': 0.7392522039663074, 'colsample_bytree': 0.7180679633740015, 'gam

### Best Model: XGBOOST

In [1]:
import pandas as pd

df = pd.read_csv("results/xgb_predictions.csv")

print(df.shape)
print(df.columns)
df.head(10)

(625783, 30)
Index(['Flow_Duration', 'Dst_Port', 'Flow_IAT_Min', 'Flow_IAT_Mean',
       'Flow_IAT_Max', 'Flow_Pkts/s', 'Flow_Byts/s', 'Pkt_Len_Max',
       'Pkt_Len_Mean', 'Pkt_Size_Avg', 'Pkt_Len_Min', 'Pkt_Len_Std',
       'Idle_Min', 'Idle_Mean', 'Idle_Max', 'Fwd_Pkts/s', 'Bwd_Pkts/s',
       'Init_Bwd_Win_Byts', 'ACK_Flag_Cnt', 'SYN_Flag_Cnt', 'Bwd_Header_Len',
       'Fwd_Pkt_Len_Max', 'Fwd_Pkt_Len_Min', 'Bwd_Pkt_Len_Max',
       'Bwd_Pkt_Len_Mean', 'y_true', 'y_pred', 'confidence', 'fold', 'model'],
      dtype='object')


,Flow_Duration,Dst_Port,Flow_IAT_Min,Flow_IAT_Mean,Flow_IAT_Max,Flow_Pkts/s,Flow_Byts/s,Pkt_Len_Max,Pkt_Len_Mean,Pkt_Size_Avg,...,Bwd_Header_Len,Fwd_Pkt_Len_Max,Fwd_Pkt_Len_Min,Bwd_Pkt_Len_Max,Bwd_Pkt_Len_Mean,y_true,y_pred,confidence,fold,model
0,5310,554,1056.0,2655.000000,4254.0,564.971751,0.000000e+00,0.0,0.000000,0.000000,...,44,0.0,0.0,0.0,0.000000,0,0,0.999963,1,XGBoost
1,141,9020,70.0,70.500000,71.0,21276.595745,1.990071e+07,1388.0,1048.500000,1398.000000,...,96,0.0,0.0,1388.0,935.333333,8,4,0.422501,1,XGBoost
2,157,443,74.0,78.500000,83.0,19108.280255,0.000000e+00,0.0,0.000000,0.000000,...,32,0.0,0.0,0.0,0.000000,4,4,0.525738,1,XGBoost
3,6799,554,6799.0,6799.000000,6799.0,294.160906,0.000000e+00,0.0,0.000000,0.000000,...,44,0.0,0.0,0.0,0.000000,0,0,0.999963,1,XGBoost
4,165,43238,165.0,165.000000,165.0,12121.212121,1.746667e+07,1441.0,1441.000000,2161.500000,...,32,1441.0,1441.0,1441.0,1441.000000,5,2,0.333489,1,XGBoost
5,40,8899,1.0,4.000000,7.0,275000.000000,8.800000e+06,32.0,32.000000,34.909091,...,8,32.0,32.0,32.0,32.000000,5,5,0.999961,1,XGBoost
6,60431,44144,121.0,5493.727273,43394.0,198.573580,1.121444e+05,1388.0,605.692308,656.166667,...,224,1388.0,1041.0,1097.0,156.714286,0,0,0.999756,1,XGBoost
7,120,49784,120.0,120.000000,120.0,16666.666667,1.181667e+07,1388.0,935.333333,1403.000000,...,32,30.0,30.0,1388.0,1388.000000,6,6,0.991469,1,XGBoost
8,199,10101,199.0,199.000000,199.0,10050.251256,7.396985e+06,1430.0,967.333333,1451.000000,...,8,42.0,42.0,1430.0,1430.000000,2,5,0.368035,1,XGBoost
9,124,9020,124.0,124.000000,124.0,16129.032258,0.000000e+00,0.0,0.000000,0.000000,...,32,0.0,0.0,0.0,0.000000,7,8,0.431448,1,XGBoost
